### Heart Diease Prediction model


In [9]:
# Import Data Manipulation Libaries
import pandas as pd
import numpy as np

# Import Data visualization Libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Import Filter warning Libraries
import warnings 
warnings.filterwarnings('ignore')

# Import scikit-learn libraries
from sklearn.preprocessing import RobustScaler , MinMaxScaler , LabelEncoder ,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report

# Import Machine Learning Model Libaries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier ,GradientBoostingClassifier , AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

# Import Neccessary libraries
import optuna
import researchpy as rp

In [6]:
# Data Ingestion

filepath = 'https://raw.githubusercontent.com/shivamsingh-itds/Heartdisease_predictionmodel/refs/heads/main/data/raw/heart.csv'
target = 'target'
df = pd.read_csv(filepath)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:
# Segregate Numerical & Categorical Columns
numerical_col = df.select_dtypes(exclude='object').columns
categorical_col = df.select_dtypes(include='object').columns


In [18]:
# Descriptive Stats
from collections import OrderedDict

# Descriptive stats

def descriptive_stats():
    numerical_col = df.select_dtypes(exclude = 'object').columns
    categorica_col = df.select_dtypes(include = 'object').columns
    num_stats = []
    cat_stats = []
    data_info = []

    for i in numerical_col:
        Q1 = df[i].quantile(0.25)
        Q3 = df[i].quantile(0.75)
        IQR = Q3 - Q1 
        LF = Q1 - 1.5*IQR
        UF = Q3 + 1.5*IQR

        outlier_count = len(df[(df[i] < LF) | (df[i] > UF)])
        outlier_percentage = outlier_count / len(df[i]) * 100

        numerical_stats = OrderedDict({
            "Feature " : i ,
            "Q1" : Q1,
            "Q3" : Q3,
            "IQR" : IQR,
            "LF" : LF,
            "UF" : UF,
            "Mean" : df[i].mean(),
            "Median" : df[i].median(),
            "Min" : df[i].min(),
            "Max" : df[i].max(),
            "Outlier count" : outlier_count,
            "outlier percentage" : outlier_percentage,
            "standard derivation": df[i].std(),
            "variance" : df[i].var(),
            "skewness" : df[i].skew(),
            "kurtosis" : df[i].kurtosis()
        })
        num_stats.append(numerical_stats)
    numerical_stats_report = pd.DataFrame(num_stats)

    for i in categorica_col:
        categorical_stats = OrderedDict({
            "Feature" : i , 
            "Unquie count" : df[i].nunique(),
            "Value count" : df[i].value_counts(),
            "mode" : df[i].mode()
        })
        cat_stats.append(categorical_stats)
    categorical_stats_report = pd.DataFrame(cat_stats)


    for i in df.columns : 
        data1 = OrderedDict({
            "Feature" : i ,
            "Missing value" : df[i].isnull().sum(),
            "Unqiue value" : df[i].nunique(),
            "value count " : df[i].value_counts().to_dict()
        })
        data_info.append(data1)
    data_info_report = pd.DataFrame(data_info)

    return categorical_stats_report,numerical_stats_report,data_info_report

categorical_stats_report,numerical_stats_report,data_info_report = descriptive_stats()

In [19]:
# Numerical stats
numerical_stats_report

,Feature,Q1,Q3,IQR,LF,UF,Mean,Median,Min,Max,Outlier count,outlier percentage,standard derivation,variance,skewness,kurtosis
0,age,47.5,61.0,13.5,27.25,81.25,54.366337,55.0,29.0,77.0,0,0.000000,9.082101,82.484558,-0.202463,-0.542167
1,sex,0.0,1.0,1.0,-1.50,2.50,0.683168,1.0,0.0,1.0,0,0.000000,0.466011,0.217166,-0.791335,-1.382961
2,cp,0.0,2.0,2.0,-3.00,5.00,0.966997,1.0,0.0,3.0,0,0.000000,1.032052,1.065132,0.484732,-1.193071
3,trestbps,120.0,140.0,20.0,90.00,170.00,131.623762,130.0,94.0,200.0,9,2.970297,17.538143,307.586453,0.713768,0.929054
4,chol,211.0,274.5,63.5,115.75,369.75,246.264026,240.0,126.0,564.0,5,1.650165,51.830751,2686.426748,1.143401,4.505423
5,fbs,0.0,0.0,0.0,0.00,0.00,0.148515,0.0,0.0,1.0,45,14.851485,0.356198,0.126877,1.986652,1.959678
6,restecg,0.0,1.0,1.0,-1.50,2.50,0.528053,1.0,0.0,2.0,0,0.000000,0.525860,0.276528,0.162522,-1.362673
7,thalach,133.5,166.0,32.5,84.75,214.75,149.646865,153.0,71.0,202.0,1,0.330033,22.905161,524.646406,-0.537410,-0.061970
8,exang,0.0,1.0,1.0,-1.50,2.50,0.326733,0.0,0.0,1.0,0,0.000000,0.469794,0.220707,0.742532,-1.458317
9,oldpeak,0.0,1.6,1.6,-2.40,4.00,1.039604,0.8,0.0,6.2,5,1.650165,1.161075,1.348095,1.269720,1.575813


In [21]:
# Data Info 
data_info_report

,Feature,Missing value,Unqiue value,value count
0,age,0,41,"{58: 19, 57: 17, 54: 16, 59: 14, 52: 13, 51: 1..."
1,sex,0,2,"{1: 207, 0: 96}"
2,cp,0,4,"{0: 143, 2: 87, 1: 50, 3: 23}"
3,trestbps,0,49,"{120: 37, 130: 36, 140: 32, 110: 19, 150: 17, ..."
4,chol,0,152,"{204: 6, 234: 6, 197: 6, 212: 5, 269: 5, 254: ..."
5,fbs,0,2,"{0: 258, 1: 45}"
6,restecg,0,3,"{1: 152, 0: 147, 2: 4}"
7,thalach,0,91,"{162: 11, 163: 9, 160: 9, 173: 8, 152: 8, 172:..."
8,exang,0,2,"{0: 204, 1: 99}"
9,oldpeak,0,40,"{0.0: 99, 1.2: 17, 1.0: 14, 0.6: 14, 0.8: 13, ..."


In [17]:
rp.codebook(df)


Variable: age    Data Type: int64 

 Number of Obs.: 303 
 Number of missing obs.: 0 
 Percent missing: 0.0 
 Number of unique values: 41 

 Range: [29, 77] 
 Mean: 54.37 
 Standard Deviation: 9.08 
 Mode: 58 
 10th Percentile: 42.0 
 25th Percentile: 47.5 
 50th Percentile: 55.0 
 75th Percentile: 61.0 
 90th Percentile: 66.0 





Variable: sex    Data Type: int64 

 Number of Obs.: 303 
 Number of missing obs.: 0 
 Percent missing: 0.0 
 Number of unique values: 2 

 Range: [0, 1] 
 Mean: 0.68 
 Standard Deviation: 0.47 
 Mode: 1 
 10th Percentile: 0.0 
 25th Percentile: 0.0 
 50th Percentile: 1.0 
 75th Percentile: 1.0 
 90th Percentile: 1.0 





Variable: cp    Data Type: int64 

 Number of Obs.: 303 
 Number of missing obs.: 0 
 Percent missing: 0.0 
 Number of unique values: 4 

 Range: [0, 3] 
 Mean: 0.97 
 Standard Deviation: 1.03 
 Mode: 0 
 10th Percentile: 0.0 
 25th Percentile: 0.0 
 50th Percentile: 1.0 
 75th Percentile: 2.0 
 90th Percentile: 2.0 





Variable: trestb